In [1]:
from pathlib import Path


def data_dir() -> Path:
    """หาโฟลเดอร์ datasets/ โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "datasets").exists():
            return p / "datasets"
    raise SystemExit(
        "ไม่พบโฟลเดอร์ datasets/ — เปิด JupyterLab จาก repo root (uv run "
        "jupyter lab)"
    )


DATA = data_dir()


# Exercise — Module 2: Feature Engineering

**Dataset (โปรดอ่านก่อนใช้งาน):** *Combined Cycle Power Plant* — Tüfekci, P. & Kaya, H. (2014). UCI ML Repository. https://doi.org/10.24432/C5002N (CC BY 4.0) — `datasets/ccpp_power_plant.csv`

โจทย์: สร้าง preprocessing pipeline ที่สมบูรณ์และป้องกัน leakage ด้วยมือคุณเอง เติมช่อง `# TODO` คำใบ้อยู่ใต้แต่ละข้อ เฉลยใน `solution.ipynb`

---

**เวลาที่คาดหวังสำหรับแบบฝึกหัดนี้:** ~30 นาที (ทำในห้องช่วง lab — เฉลยเปิดเผยหลังจบช่วง)


In [2]:
import polars as pl  # noqa: E402

ccpp = pl.read_csv(DATA / "ccpp_power_plant.csv")
X = ccpp.select("AT", "V", "AP", "RH")
y = ccpp["PE"]


## ข้อ 1 — split ก่อนแตะทุกอย่าง

แบ่ง train/test 80/20 ด้วย `random_state=42`

TODO

In [3]:
from sklearn.model_selection import train_test_split  # noqa: E402, F401

X_train, X_test, y_train, y_test = ...


TypeError: cannot unpack non-iterable ellipsis object

## ข้อ 2 — pipeline พื้นฐาน: scaler + ridge

สร้าง `Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=1.0))])` fit กับ train แล้วรายงาน R² ทั้ง train และ test

TODO

In [4]:
from sklearn.pipeline import Pipeline  # noqa: E402, F401
from sklearn.preprocessing import StandardScaler  # noqa: E402, F401
from sklearn.linear_model import Ridge  # noqa: E402, F401

pipe = ...
pipe.fit(...)
r2_train = ...
r2_test = ...
print(f"train R² = {r2_train:.4f} | test R² = {r2_test:.4f}")


AttributeError: 'ellipsis' object has no attribute 'fit'

**คำใบ้:** `pipe.score(X_train, y_train)` และ `pipe.score(X_test, y_test)`

## ข้อ 3 — เพิ่ม polynomial feature

แก้ pipeline เป็น scaler → `PolynomialFeatures(degree=2, include_bias=False)` → ridge แล้วเทียบ test R² กับข้อ 2
(อย่าลืม: PolynomialFeatures หลัง scaler — เพราะอะไร? คิดเรื่องขนาดของค่ายกกำลัง)

TODO

In [5]:
from sklearn.preprocessing import PolynomialFeatures  # noqa: E402, F401

pipe_poly = ...
...
r2_poly = ...
print(f"poly(2)+ridge test R² = {r2_poly:.4f}")


TypeError: unsupported format string passed to ellipsis.__format__

## ข้อ 4 — เลือก alpha ด้วย cross-validation

ใช้ `GridSearchCV` ค้นหา `ridge__alpha` ใน [0.01, 0.1, 1, 10, 100] ด้วย 5-fold CV บน train set แล้วรายงาน alpha ที่ดีที่สุดและ test R² ของโมเดลสุดท้าย

TODO

In [6]:
from sklearn.model_selection import GridSearchCV  # noqa: E402, F401

param_grid = {"ridge__alpha": ...}
search = ...
search.fit(...)
best_alpha = ...
r2_best = ...
print(f"best alpha = {best_alpha} | test R² = {r2_best:.4f}")


AttributeError: 'ellipsis' object has no attribute 'fit'

**คำใบ้:** ตั้งชื่อพารามิเตอร์ด้วยชื่อขั้นใน pipeline: `"ridge__alpha"`

## ข้อ 5 — domain feature ด้วยมือ + เทียบ

เพิ่มคอลัมน์ `AT_x_RH` (= AT × RH) และ `dew_point` (ฟังก์ชัน Magnus จาก book ข้อ 6) เข้าไปใน X_train/X_test
แล้วรัน pipeline ของข้อ 3 (scaler → poly(2) → ridge ที่ alpha ดีสุด) กับ feature set ใหม่ — test R² ดีขึ้นไหม?

TODO: สร้าง X2_train, X2_test ที่มี AT, V, AP, RH, AT_x_RH, dew_point

TODO: fit pipeline เดิม (ใช้ alpha ที่ดีสุดจากข้อ 4) แล้วรายงาน test R²

In [7]:
import numpy as np  # noqa: E402


def dew_point(at, rh):
    a, b = 17.62, 243.12
    gamma = np.log(rh / 100) + a * at / (b + at)
    return b * gamma / (a - gamma)


X2_train = ...
X2_test = ...

r2_domain = ...
print(f"poly + domain features test R² = {r2_domain:.4f}")


TypeError: unsupported format string passed to ellipsis.__format__

## ข้อ 6 — โจทย์คิด (ตอบใน markdown)

1. ถ้าเรา fit `StandardScaler` กับข้อมูล **ทั้งชุด** (รวม test) ก่อน split — ผล test R² จะเปลี่ยนไปอย่างไร และทำไมจึงถือว่าโกง?
2. ถ้าเปลี่ยนโมเดลจาก Ridge เป็น RandomForest — ยังต้อง scale ไหม เพราะอะไร?

คำตอบข้อ 6:

1. (พิมพ์คำตอบ)
2. (พิมพ์คำตอบ)